<a href="https://colab.research.google.com/github/badaozhiwei-cmyk/zero-to-herO/blob/main/try.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 下载数据集（如果还没下载的话）
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# 2. 读取文本数据
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"文本总长度: {len(text)} 字符")

# 3. 看看前 200 个字符长啥样
print("--- 前 200 个字符 ---")
print(text[:200])


--2026-03-18 13:20:31--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.009s  

2026-03-18 13:20:31 (115 MB/s) - ‘input.txt’ saved [1115394/1115394]

文本总长度: 1115394 字符
--- 前 200 个字符 ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [ ]:
#找出所有不重复的字符（这就是我们的“原始词表”）
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"\n词表大小 (Vocab Size): {vocab_size}")
print(f"\n字符有 {' '.join(chars)}")


词表大小 (Vocab Size): 65

字符有 
   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z


In [ ]:
stoi = {s : i for i,s in enumerate (chars)}
itos = {i : s for s,i in stoi.items()}
#列表推导式
encode =  lambda s : [stoi[idx] for idx in s]
decode =  lambda i : ''.join([itos[idx] for idx in i])
test_str = "hii there"
encoded_val = encode(test_str)
print(f"编码后: {encoded_val}")
print(f"解码后: {decode(encoded_val)}")
import torch
#Python 列表在内存里是分散的，而 Tensor 是连续存放的
data = torch.tensor(encode(text),dtype=torch.long)
print(data[:10])

编码后: [46, 47, 47, 1, 58, 46, 43, 56, 43]
解码后: hii there
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [ ]:
block_size = 8
x = data[   : block_size]
y = data[ 1 : block_size+1]
for t in range(block_size):
  context = x[ : t+1]
  target = y[t]
  print(f"输入 {context} 时，目标是: {target}")
#y领先x一个身位
print(x[0],y[0])

输入 tensor([18]) 时，目标是: 47
输入 tensor([18, 47]) 时，目标是: 56
输入 tensor([18, 47, 56]) 时，目标是: 57
输入 tensor([18, 47, 56, 57]) 时，目标是: 58
输入 tensor([18, 47, 56, 57, 58]) 时，目标是: 1
输入 tensor([18, 47, 56, 57, 58,  1]) 时，目标是: 15
输入 tensor([18, 47, 56, 57, 58,  1, 15]) 时，目标是: 47
输入 tensor([18, 47, 56, 57, 58,  1, 15, 47]) 时，目标是: 58
tensor(18) tensor(47)


In [ ]:
torch.manual_seed(1337) # 保证你我的随机结果一致，方便对账
batch_size = 4 # 每次并行处理 4 个小块
block_size = 8 # 每个小块长度为 8

n = int(0.9*len(data))
train_data = data [:n]
val_data = data[n:]

#（B,T)
def get_batch(split):
  d = train_data if split == 'train' else val_data

  #100 多万字的 data 里随机插 4 个眼，每个眼往后取 8 个词
  ix = torch.randint(len(data) - block_size, (batch_size,))
  #它把 4 个一维的向量（长度 8）像叠罗汉一样叠在一起，变成了一个 (4, 8) 的二维矩阵（张量）。
  x = torch.stack([data[ i : i + block_size ]  for i in ix])
  y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
  return x , y
xbatch , ybatch = get_batch('train')
print(f"{xbatch.shape} \n {ybatch.shape}")

torch.Size([4, 8]) 
 torch.Size([4, 8])


In [ ]:
import torch.nn as nn
from torch.nn import functional as F
n_embd = 32      # 每个字符向量的长度
block_size = 8     # 模型能看到的最大前文长度
head_size = 16     # 注意力头内部的特征维度
class BigramlanguageModel( nn.Module ):
  def __init__(self,vocab_size):
    super().__init__()
    #nn.Module 的子类 ，实例化为 self.token_embd_table
    self.token_embd_table = nn.Embedding(vocab_size,vocab_size)

  def forward(self,idx,target=None):
    #B,T到B,T,C
    logits = self.token_embd_table(idx)
    if target is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      target = target.view(B*T)
      loss = F.cross_entropy(logits,target)
    return logits,loss

  def generate(self,idx,max_new_token):
    for _ in range(max_new_token):
      logits,loss = self(idx)
      #不管你之前生成了多长的一串词，我只取最后那一个词的预测结果（Logits）。
      logits = logits[:,-1,:]
      probs = F.softmax(logits,dim=-1)
      idx_next = torch.multinomial(probs,num_samples=1)
      idx = torch.cat((idx,idx_next),dim=1)
    return idx


m = BigramlanguageModel(vocab_size)
logits , loss = m(xbatch , ybatch)
print(logits.shape,loss)
#输入BT
context = torch.zeros((1,1),dtype=torch.long)#（B,T）
#【0】从这个“批次”中取出第一个（也是唯一一个）序列。
#print(decode(m.generate(context,max_new_token=100)[0].tolist()))
# 1. 先把生成的整个序列存下来
generated_idx = m.generate(context, max_new_token=100)
# 2. 打印形状
print(f"生成的张量形状: {generated_idx.shape}")
# 3. 再进行解码 预测下一个词只看当前这一个词。所以即使你生成 100 个词，它每次也只看最后那 1 个。哪怕你训练时用的 block_size 是 8，预测时它也不需要看前 8 个。
print(f"这是{decode(generated_idx[0].tolist())}")

torch.Size([32, 65]) tensor(4.6815, grad_fn=<NllLossBackward0>)
生成的张量形状: torch.Size([1, 101])
这是
lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheH


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

for steps in range(10000):
  xbatch , ybatch = get_batch("train")
  logits,loss = m(xbatch,ybatch)
  optimizer.zero_grad()
  loss.backward()      # 1. 算出梯度
  optimizer.step()     # 2. 更新参数
  if steps % 100 == 0:
    print(f"{steps}步的损失{loss.item()}")

0步的损失4.673117160797119
100步的损失4.279718399047852
200步的损失4.41204309463501
300步的损失4.287530422210693
400步的损失4.754645824432373
500步的损失4.1630167961120605
600步的损失4.184802055358887
700步的损失4.302606582641602
800步的损失3.7548255920410156
900步的损失4.232077598571777
1000步的损失3.975274085998535
1100步的损失3.9422800540924072
1200步的损失3.81945538520813
1300步的损失3.6076250076293945
1400步的损失3.8296868801116943
1500步的损失3.8712711334228516
1600步的损失3.8610360622406006
1700步的损失3.6518595218658447
1800步的损失3.7494773864746094
1900步的损失3.645418882369995
2000步的损失3.6270880699157715
2100步的损失3.4710121154785156
2200步的损失3.618513822555542
2300步的损失3.350724220275879
2400步的损失3.181983470916748
2500步的损失3.3766672611236572
2600步的损失3.4017748832702637
2700步的损失3.358638048171997
2800步的损失3.026623010635376
2900步的损失3.0370638370513916
3000步的损失3.266636610031128
3100步的损失3.139281749725342
3200步的损失3.0174720287323
3300步的损失2.843639373779297
3400步的损失3.3219804763793945
3500步的损失3.148622512817383
3600步的损失3.3589587211608887
3700步的损失2.998227119445801
3800步的损失3.26

[0]：把维度从 (1, 101) 降到 (101,)。

.tolist()：把张量转回 Python 基础列表，确保 itos[idx] 拿到的 idx 是纯整数

In [ ]:
# 1. 生产：生成 4 批，每批 100 个新词
context_4 = torch.zeros((4, 1), dtype=torch.long)
generated_idx = m.generate(context_4, max_new_token=100) # 形状变为 (4, 101)

# 2. 翻译：把这 4 批全部解码
all_results = [decode(row.tolist()) for row in generated_idx]

# 3. 展示：打印你想看的任何一批
print(f"第 2 批次的完整 100 词输出：\n{all_results[2]}")

第 2 批次的完整 100 词输出：

stil for have to im lith burn wish thergir, co-d.

MUSWARGTA:
Rawar hell but coptinver hit whan
HaSh


In [ ]:
context = torch.zeros((1,1),dtype=torch.long)
print(decode(m.generate(context,max_new_token=500)[0].tolist()))


 have of no here ar-vinfre, chull.

KCHALOS, me is, is lancove.

Rled thur claught wilr lil our you mo ther zake, on to se re chy wheeiack to bla, ith he latork leng: hast manlann do fary.

YYasty sou, No ther reainns.
 an wer dars no shin.

LUGEUCEL:
An so, thou my thou hanly mord man alle ucght swall so ecces but wher, halt sante, shis butist:
I murst. 
Hen your to dice to by no so mee houre mand sto gor gord thale for dhers will wily the theart Ther Pawind ol, wall wink there my hor Mare shou


### 自注意力机制

In [ ]:

B,T,C = 1,8,2
x = torch.randn(B,T,C)
tril = torch.tril(torch.ones(T,T))
wei = tril /tril.sum(1,keepdim=True)

out = wei @ x
wei,out.shape

(tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
         [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
         [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
         [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]]),
 torch.Size([1, 8, 2]))

In [ ]:
B,T,C = 4 , 8 , 32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C,head_size,bias=False)
query = nn.Linear(C,head_size,bias=False)
value = nn.Linear(C,head_size,bias=False)


k = key(x)
print(k.shape)
q = query(x)

wei = q @ k.transpose(-2,-1)*(head_size**-0.5)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
print(wei.shape)
wei = F.softmax(wei,dim=-1)

v = value(x)
out = wei@v
out.shape

torch.Size([4, 8, 16])
torch.Size([4, 8, 8])


torch.Size([4, 8, 16])

In [ ]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6025, 0.3975, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3149, 0.3457, 0.3393, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2408, 0.2069, 0.2138, 0.3385, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1562, 0.1965, 0.2040, 0.1430, 0.3003, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.2509, 0.1361, 0.2205, 0.1153, 0.0902, 0.0000, 0.0000],
        [0.1174, 0.1055, 0.1350, 0.1203, 0.2091, 0.1855, 0.1272, 0.0000],
        [0.1126, 0.1375, 0.1295, 0.1104, 0.1167, 0.1601, 0.1197, 0.1136]],
       grad_fn=<SelectBackward0>)

### 🪓 第一板斧：Multi-Head Attention (多头并行)

In [ ]:
class Head(nn.Module):
    """ 一个单头自注意力机制 """

    def __init__(self, head_size):
        super().__init__()
        # 所有的“零件”都必须在这里定义
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # 缓存下三角矩阵，block_size 是你之前定义的全局变量（比如 8）
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        #(T, T) 的矩阵就是“词与词之间的关系图”。第i行第j列的数字，代表了第i个词对第j个词的“关注度”
        # 计算注意力权重 (Affinity)
        # 注意：这里用 k.shape[-1] 自动获取 head_size，防止变量名未定义报错
        wei = q @ k.transpose(-2, -1) * (k.shape[-1]**-0.5)

        # 遮盖未来信息 (Mask)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        # 聚合干货信息
        v = self.value(x) # (B, T, head_size)
        out = wei @ v # (B, T, head_size)
        return out

class MultHeadAttention(nn.Module):
    """ 多头自注意力机制 """
    def __init__(self, num_heads, head_size):
      super().__init__()
      self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
      #这里的投影层是为了把拼接后的向量映射回 n_embd 维度 并且全局权重融合特征
      self.proj = nn.Linear(head_size * num_heads, n_embd)

    def forward(self,x):
      # (B, T, head_size * num_heads） -> (B, T, n_embd)
      out = torch.cat([h(x) for h in self.heads],dim=-1)
      out = self.proj(out)
      return out

### 🪓 第二板斧：FeedForward (消化层)

In [ ]:
class FeedForward(nn.Module):
  def __init__(self,n_embd):
    super().__init__()
    self.net = nn.Sequential(
      # 工业标准：中间层放大 4 倍
      nn.Linear(n_embd,4*n_embd),
      nn.ReLU(),
      # 再缩放回来
      nn.Linear(4*n_embd,n_embd),
    )
  def forward(self,x):
    return self.net(x)


### 🪓 第三板斧：Block (核心积木)

In [ ]:
class Block(nn.Module):
  def __init__(self,n_embd,n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultHeadAttention(n_head,head_size)
    self.ffwd = FeedForward(n_embd)
    #在forward进入 Attention 之前，和进入 FeedForward 之前，都要先做一次 LayerNorm
    # 加入两层归一化
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self,x):
    x = x + self.sa(x)
    x = x + self.ffwd(x)
    return x

你的设定：你在模型初始化时定义了 block_size = 8。这意味着 position_embedding_table 只有 8 个位置的向量（0 到 7）。

报错原因：在你的 generate 函数里，你让模型生成 500 个词。随着生成的进行，输入 idx 的长度会从 1 变成 2, 3... 一直增加到 100, 200...

冲突点：当 idx 的长度超过 8 时，forward 函数里的 torch.arange(T) 就会产生像 8, 9, 10... 这样的索引。但你的位置表里最高只有 7，所以它就报错“越界”了

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
n_embd = 32      # 每个字符向量的长度
block_size = 8     # 模型能看到的最大前文长度
head_size = 16     # 注意力头内部的特征维度
class BigramlanguageModel( nn.Module ):
  def __init__(self,vocab_size):
    super().__init__()
    #nn.Module 的子类 ，实例化为 self.token_embd_table
    self.token_embd_table = nn.Embedding(vocab_size,n_embd)
    self.posi_embd_table = nn.Embedding(block_size, n_embd)
    #self.SA_head = Head(n_embd)
    self.blocks = nn.Sequential(
    Block(n_embd, n_head=4),
    Block(n_embd, n_head=4),
    Block(n_embd, n_head=4), # 堆叠三层
)
    self.ln_f = nn.LayerNorm(n_embd) # final layer norm
    #通常指模型最后一层，负责将特征向量映射回词表大小（vocab size）的线性层。
    self.lm_head = nn.Linear(n_embd,vocab_size)


  def forward(self,idx,target=None):
    B,T = idx.shape

    tok_emb = self.token_embd_table(idx)
    pos_emb = self.posi_embd_table(torch.arange(T,device=idx.device))
    x = tok_emb + pos_emb
    #x = self.SA_head(x)
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x)


    if target is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      target = target.view(B*T)
      loss = F.cross_entropy(logits,target)
    return logits,loss

  def generate(self,idx,max_new_token):
    for _ in range(max_new_token):

      # 这样 idx_cond 的长度永远不会超过 8
      idx_cond = idx[:, -block_size:]

      logits,loss = self(idx_cond)
      #不管你之前生成了多长的一串词，我只取最后那一个词的预测结果（Logits）。
      logits = logits[:,-1,:]
      probs = F.softmax(logits,dim=-1)
      idx_next = torch.multinomial(probs,num_samples=1)
      idx = torch.cat((idx,idx_next),dim=1)
    return idx


m = BigramlanguageModel(vocab_size)
logits , loss = m(xbatch , ybatch)
print(logits.shape,loss)


torch.Size([32, 65]) tensor(4.1713, grad_fn=<NllLossBackward0>)


In [ ]:
# 1. 依然使用 AdamW，但可以稍微调大一点学习率，或者增加步数
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

for steps in range(5000): # 咱们直接练 5000 步！
    # 采样
    xb, yb = get_batch('train')

    # 前向传播
    logits, loss = m(xb, yb)

    # 反向传播三部曲
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % 500 == 0:
        print(f"步数 {steps}: 损失值 {loss.item():.4f}")

# 训练完后，看看它写出的“进化版”莎士比亚
context = torch.zeros((1, 1), dtype=torch.long)
print("\n--- 训练后的生成结果 ---")
print(decode(m.generate(context, max_new_token=500)[0].tolist()))

步数 0: 损失值 4.1960
步数 500: 损失值 2.7419
步数 1000: 损失值 2.6679
步数 1500: 损失值 2.2116
步数 2000: 损失值 2.7303
步数 2500: 损失值 2.8258
步数 3000: 损失值 2.1873
步数 3500: 损失值 2.0320
步数 4000: 损失值 2.3030
步数 4500: 损失值 2.0805

--- 训练后的生成结果 ---

Thud;.

TG SEs Reres yadour to singt hulvenat ,urourbures to ustof codor lich sinter
Thostedser,
Muecioncop-with tis ands such bust-t she urear, apo preuis boroy-ant ating yinv ylromy
Bundd I thee mothey! jurd ast an piow hows do and it yoouch an youth sancuth.

AMOY! Sint of if dan,
Sad biden.

SIONLUS:
Ay ave if be,
Haty is grot.

ANG: asbun vou mo I
a frros alold.
A fonds wyoun praant, himm your rodevi'lloud dos theens; foll birkJhe atitins on; Orat andipleamer lemy uthou uns waster
poth mis 
